In [ ]:
# ==========================================
# CELL 1: Data Loading & Preprocessing
# ==========================================

# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import warnings
from pathlib import Path

# Display warnings so that convergence and compatibility issues remain visible.
warnings.filterwarnings("default")

# ==========================================
# --- GLOBAL VISUALIZATION SETTINGS ---
# (Adjust these parameters to change fonts across all plots)
# ==========================================
plt.rcParams['font.family'] = 'Arial'     # Font family (e.g., 'Arial', 'Times New Roman')
plt.rcParams['font.size'] = 12            # Base font size
plt.rcParams['axes.unicode_minus'] = False # Fix minus sign display
sns.set_style("whitegrid")                # Plot style

# Set figure size (width, height) in inches
plt.rcParams['figure.figsize'] = (12, 8)

# ==========================================
# 1. Reproducibility and repository paths
# ==========================================
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "Data.csv"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
TARGET_COLUMN = "RHg"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Data file not found: {DATA_PATH}. Run the notebook from the repository root "
        "and place Data.csv in the data directory."
    )

data = pd.read_csv(DATA_PATH)
print(f"Success: data loaded from '{DATA_PATH}'")

# ==========================================
# 2. Basic Data Inspection
# ==========================================
print("\n--- Dataset Info ---")
print("Shape:", data.shape)
print("\nColumns:", data.columns.tolist())

# Validate the curated dataset before analysis
missing_by_column = data.isna().sum()
missing_values = int(missing_by_column.sum())
print(f"\nTotal Missing Values: {missing_values}")
if missing_values:
    raise ValueError(
        "The curated dataset is expected to be complete. Missing values were found in: "
        f"{missing_by_column[missing_by_column > 0].to_dict()}"
    )

duplicate_rows = int(data.duplicated().sum())
print(f"Exact Duplicate Rows: {duplicate_rows}")
if duplicate_rows:
    warnings.warn(
        f"The processed dataset contains {duplicate_rows} exact duplicate rows. "
        "Verify their provenance before model training to avoid train/test overlap."
    )

if TARGET_COLUMN not in data.columns:
    raise KeyError(f"Target column '{TARGET_COLUMN}' is not present in Data.csv")

# Basic statistics
print("\n--- Basic Statistics ---")
print(data.describe())

# ==========================================
# 3. Feature/Target Separation & Outlier Detection
# ==========================================
# Separate features (X) and target (y) by name rather than column position.
# Raw physical values are retained for tree-based models.
X = data.drop(columns=[TARGET_COLUMN])
y = data[TARGET_COLUMN]

non_numeric_features = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric_features:
    raise TypeError(f"Non-numeric input features detected: {non_numeric_features}")

print(f"\nFeatures (X): {X.shape}")
print(f"Target (y): {y.shape}")

# Simple Outlier Detection (Z-score method)
print("\n--- Outlier Detection (> 3 Std Dev) ---")
for col in X.columns:
    mean_val = X[col].mean()
    std_val = X[col].std()
    # Avoid division by zero if std is 0
    if std_val > 0:
        outliers = X[(X[col] < mean_val - 3*std_val) | (X[col] > mean_val + 3*std_val)]
        if len(outliers) > 0:
            print(f"Feature '{col}': {len(outliers)} potential outliers detected")

# ==========================================
# 4. Standardization (FOR VISUALIZATION ONLY)
# ==========================================
# We create a separate scaled dataframe specifically for EDA (Exploratory Data Analysis).
# This ensures visual comparisons (like violin plots) are on the same scale.
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled_array, columns=X.columns)

print("\n--- Preprocessing Status ---")
print("1. Raw Data (X): Preserved for model training (interpretable physical units).")
print("2. Scaled Data (X_scaled_df): Created for visualization purposes only.")
print(f"Scaled Data Range: Min={X_scaled_df.min().min():.2f}, Max={X_scaled_df.max().max():.2f}")


In [ ]:
# ==========================================
# CELL 2: Exploratory Data Analysis (EDA) - FINAL ADJUSTMENT
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import pandas as pd

# --- Visualization Parameters ---
TITLE_SZ = 16       # Title font size
LABEL_SZ = 14       # Axis label font size
TICK_SZ  = 14       # Tick label font size (Large enough for publication)
DPI      = 300      # High resolution

# ==========================================
# 1. Feature Distribution (Violin Plots)
# ==========================================
# Create a copy for visualization
X_vis = X_scaled_df.copy()

# --- SCI Standard Fix for Unit Labels ---
# We use simple string replacement to fix the double parenthesis issue.
# We use the Unicode degree symbol '°' (U+00B0) instead of LaTeX.
# This ensures the unit font matches the Arial bold title font naturally.
new_columns = {}
for col in X_vis.columns:
    if '℃' in col:
        # Simply replace the character, do not add extra brackets
        new_columns[col] = col.replace('℃', '°C')
    else:
        new_columns[col] = col

X_vis.rename(columns=new_columns, inplace=True)

# Dynamic layout
n_features = len(X_vis.columns)
n_cols = 7
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(22, 3.8 * n_rows))
axes = axes.ravel()

print("Generating Violin Plots (SCI Standard Labels)...")

for i, col in enumerate(X_vis.columns):
    sns.violinplot(y=X_vis[col], ax=axes[i], color='skyblue')
    
    # Title settings
    axes[i].set_title(col, fontsize=TITLE_SZ, fontweight='bold')
    axes[i].set_ylabel('')
    
    # Tick settings - BOLD and LARGE
    axes[i].tick_params(axis='both', which='major', labelsize=TICK_SZ)
    for label in axes[i].get_yticklabels():
        label.set_fontweight('bold')
        
    axes[i].grid(True, alpha=0.3)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_distribution_violin.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 2. Target Variable Analysis
# ==========================================
print("Analyzing Target Variable...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Helper function for consistent style
def apply_sci_style(ax, title, xlabel=None, ylabel=None):
    ax.set_title(title, fontsize=TITLE_SZ, fontweight='bold')
    if xlabel: ax.set_xlabel(xlabel, fontsize=LABEL_SZ, fontweight='bold')
    if ylabel: ax.set_ylabel(ylabel, fontsize=LABEL_SZ, fontweight='bold')
    ax.tick_params(axis='both', labelsize=TICK_SZ)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.grid(True, alpha=0.3)

# A. Histogram
axes[0,0].hist(y, bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
apply_sci_style(axes[0,0], 'Histogram', ylabel='Frequency')

# B. Box Plot
sns.boxplot(y=y, ax=axes[0,1], color='lightgreen')
apply_sci_style(axes[0,1], 'Box Plot')

# C. Q-Q Plot
stats.probplot(y, dist="norm", plot=axes[1,0])
apply_sci_style(axes[1,0], 'Q-Q Plot', xlabel='Theoretical Quantiles', ylabel='Ordered Values')
axes[1,0].get_lines()[0].set_color('blue')
axes[1,0].get_lines()[1].set_color('red')

# D. Density Plot
y.plot.density(ax=axes[1,1], color='purple', linewidth=2)
apply_sci_style(axes[1,1], 'Density Plot', ylabel='Density')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'target_variable_analysis.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 3. Feature Statistics Summary
# ==========================================
print("Summarizing Feature Statistics...")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

def plot_stat_bar(ax, data, title, color):
    data = data.sort_values(ascending=True)
    
    # Sync index names with X_vis to maintain the fixed "°C" label
    # This ensures the bar chart labels match the violin plot titles
    fixed_index = [name.replace('℃', '°C') if '℃' in name else name for name in data.index]
    
    ax.barh(fixed_index, data.values, color=color, alpha=0.8)
    ax.set_title(title, fontsize=TITLE_SZ, fontweight='bold')
    ax.tick_params(axis='both', labelsize=TICK_SZ)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    ax.grid(True, alpha=0.3)

# Use X_scaled_df stats but apply the label fix inside the plotting function
plot_stat_bar(axes[0,0], X_scaled_df.mean(), 'Feature Means (Standardized)', 'steelblue')
plot_stat_bar(axes[0,1], X_scaled_df.std(),  'Feature Standard Deviations', 'orange')
plot_stat_bar(axes[1,0], X_scaled_df.skew(), 'Feature Skewness', 'green')
plot_stat_bar(axes[1,1], X_scaled_df.kurtosis(), 'Feature Kurtosis', 'red')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'feature_statistics_summary.png', dpi=DPI, bbox_inches='tight')
plt.show()


In [ ]:
# ==========================================
# CELL 3: Pearson Correlation Analysis (Fixed Grid)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.ticker as ticker

# --- Visualization Parameters ---
TITLE_SZ = 22       # Main title
LABEL_SZ = 16       # Axis labels
TICK_SZ  = 15       # Feature labels
ANNOT_SZ = 11       # Annotation text size
DPI      = 300      # Resolution

# ==========================================
# 1. Data Preparation
# ==========================================
corr_data = X_scaled_df.copy()
corr_data['RHg'] = y.values

# Fix Unit Labels (℃ -> °C)
new_cols = {col: col.replace('℃', '°C') for col in corr_data.columns if '℃' in col}
corr_data.rename(columns=new_cols, inplace=True)

# Calculate Matrix
correlation_matrix = corr_data.corr(method='pearson')

# Save CSV
correlation_matrix.to_csv(RESULTS_DIR / 'pearson_matrix.csv', encoding='utf-8')

# ==========================================
# Figure 1: Combined View (Full Fill)
# ==========================================
print("Generating Figure 1: Combined View...")
fig1, axes = plt.subplots(1, 2, figsize=(28, 14), gridspec_kw={'width_ratios': [1.4, 1]})

# Left: Full Heatmap
sns.heatmap(correlation_matrix, mask=None, cmap='RdBu_r', center=0, vmax=1, vmin=-1,
            annot=True, fmt='.2f', square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            ax=axes[0], annot_kws={"size": 9, "weight": "bold"})
axes[0].set_title('Pearson Correlation Matrix (Full)', fontsize=TITLE_SZ, fontweight='bold', pad=20)
axes[0].tick_params(labelsize=TICK_SZ)
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right', fontweight='bold')
plt.setp(axes[0].get_yticklabels(), fontweight='bold')

# Right: Bar Chart
target_corr = correlation_matrix['RHg'].drop('RHg').sort_values(ascending=True)
colors = ['#d62728' if v < 0 else '#4169E1' for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors, alpha=0.8)
axes[1].axvline(x=0, color='black', linewidth=1)
axes[1].set_title('Correlation with Mercury Removal Efficiency', fontsize=TITLE_SZ, fontweight='bold')
axes[1].tick_params(labelsize=TICK_SZ)
for label in axes[1].get_yticklabels() + axes[1].get_xticklabels():
    label.set_fontweight('bold')
axes[1].grid(True, alpha=0.3, axis='x')
# Add values
for i, v in enumerate(target_corr.values):
    offset = 0.01 if v >= 0 else -0.01
    ha = 'left' if v >= 0 else 'right'
    axes[1].text(v + offset, i, f'{v:.3f}', va='center', ha=ha, fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pearson_combined_full.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# Figure 2: Large Full Heatmap (Reference Style)
# ==========================================
print("Generating Figure 2: Large Full Heatmap...")
fig2, ax2 = plt.subplots(figsize=(16, 14))

sns.heatmap(correlation_matrix, mask=None, cmap='RdBu_r', center=0, vmax=1, vmin=-1,
            annot=True, fmt='.2f', square=True, linewidths=1.5, linecolor='white',
            cbar_kws={"shrink": 0.8}, ax=ax2, annot_kws={"size": ANNOT_SZ, "weight": "bold"})

ax2.set_title('Pearson Correlation Matrix (All Values)', fontsize=TITLE_SZ+4, fontweight='bold', pad=20)
plt.setp(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=TICK_SZ, fontweight='bold')
plt.setp(ax2.get_yticklabels(), fontsize=TICK_SZ, fontweight='bold')
ax2.set_xlabel('')
ax2.set_ylabel('')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pearson_matrix_large_full.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# Figure 3: Precise Hybrid Heatmap (FIXED GRID)
# ==========================================
print("Generating Figure 3: Precise Hybrid Heatmap (No Cross Lines)...")
fig3, ax3 = plt.subplots(1, 1, figsize=(16, 14))

# 1. Draw Heatmap Image
im = ax3.imshow(correlation_matrix.values, cmap='RdBu_r', aspect='auto', 
               vmin=-1, vmax=1, interpolation='nearest')

# 2. CRITICAL FIX: Disable the default Major Grid (the cross lines through numbers)
ax3.grid(False) 

# 3. Add Custom Minor Grid (The borders between cells)
rows, cols = correlation_matrix.shape
ax3.set_xticks(np.arange(cols+1)-.5, minor=True)
ax3.set_yticks(np.arange(rows+1)-.5, minor=True)
ax3.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax3.tick_params(which="minor", size=0) # Hide minor tick marks

# 4. Add Numbers (Lower Triangle Only)
for i in range(rows):
    for j in range(cols):
        val = correlation_matrix.iloc[i, j]
        
        # Adaptive text color
        text_color = "white" if abs(val) > 0.6 else "black"
        
        if i > j:  # Lower Triangle
            ax3.text(j, i, f'{val:.2f}', ha="center", va="center", 
                     color=text_color, fontsize=ANNOT_SZ, fontweight='bold')
        elif i == j:  # Diagonal
            ax3.text(j, i, f'{val:.2f}', ha="center", va="center", 
                     color="white", fontsize=ANNOT_SZ, fontweight='bold')

# 5. Axis Labels & Formatting
ax3.set_xticks(np.arange(cols))
ax3.set_yticks(np.arange(rows))
ax3.set_xticklabels(correlation_matrix.columns, rotation=45, ha='right', fontweight='bold', fontsize=TICK_SZ)
ax3.set_yticklabels(correlation_matrix.columns, fontweight='bold', fontsize=TICK_SZ)

# Colorbar
cbar = plt.colorbar(im, ax=ax3, shrink=0.8)
cbar.locator = ticker.MultipleLocator(0.2)
cbar.update_ticks()
cbar.ax.tick_params(labelsize=TICK_SZ)

ax3.set_title('Pearson Correlation Matrix (Hybrid View)', fontsize=TITLE_SZ+4, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'pearson_correlation_matrix_precise.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 4. Text & CSV Export (Keep existing logic)
# ==========================================
print("Generating distribution data...")
target_corr.to_csv(RESULTS_DIR / 'target_correlation_data.csv')

# High Correlation Pairs
print("\n--- High Correlation Pairs (|r| > 0.7) ---")
high_corr_pairs = []
for i in range(rows):
    for j in range(i+1, cols):
        val = correlation_matrix.iloc[i, j]
        if abs(val) > 0.7:
            high_corr_pairs.append((correlation_matrix.columns[i], 
                                  correlation_matrix.columns[j], 
                                  val))

high_corr_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for pair in high_corr_pairs:
    print(f"{pair[0]} - {pair[1]}: {pair[2]:.3f}")

print("\nAll tasks completed.")


In [ ]:
# ==========================================
# CELL 4: Model Screening & Training (Leakage-Free)
# ==========================================

from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
import time
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("default")

# ==========================================
# 1. Data Splitting (Standard 80/20 Split)
# ==========================================
# NOTE: We use a standard 80/20 split for initial screening.
# Sensitivity analysis on split ratios will be performed in a later step.
print("1. Splitting Data (80% Train, 20% Test)...")

# CRITICAL: Split Raw Data FIRST to prevent leakage
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)

print(f"   Training Samples: {X_train.shape[0]}")
print(f"   Testing Samples:  {X_test.shape[0]}")

# ==========================================
# 2. Conditional Preprocessing
# ==========================================
print("2. Applying Conditional Preprocessing...")

# A. Standardization (For Linear/Distance models ONLY)
# Fit on TRAIN, transform TEST
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# B. Raw Data (For Tree-based models)
# X_train and X_test are already ready.

print("   - Scaled Data: Prepared for ElasticNet & SVR")
print("   - Raw Data: Prepared for RF, XGB, CatBoost, etc. (Preserves physical meaning)")

# ==========================================
# 3. Model Definition
# ==========================================
# Group 1: Models requiring Scaled Data
models_scaled = {
    'ElasticNet': ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_SEED),
    'SVR': SVR(kernel='rbf', C=100, gamma='scale'),
}

# Group 2: Models using Raw Data
models_raw = {
    'LightGBM': lgb.LGBMRegressor(n_estimators=100, random_state=RANDOM_SEED, verbose=-1),
    'GBR': GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_SEED),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=RANDOM_SEED, eval_metric='rmse'),
    'CatBoost': CatBoostRegressor(iterations=100, random_state=RANDOM_SEED, verbose=0, allow_writing_files=False),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
}

# Combine dictionaries for iteration
all_models = {**models_scaled, **models_raw}

# Containers for results
metrics_list = []
predictions_dict = {'Actual': y_test.values} # Start with actual values

# ==========================================
# 4. Training Loop
# ==========================================
print("\n3. Training Models...")
print("-" * 85)
print(f"{'Model':<15} | {'RMSE':<10} | {'MAE':<10} | {'R2':<10} | {'Time (s)':<10}")
print("-" * 85)

results = {} # Keep for immediate use in next cell if needed

for name, model in all_models.items():
    start_time = time.time()
    
    # Select Data based on model type
    if name in models_scaled:
        X_tr, X_te = X_train_scaled, X_test_scaled
    else:
        X_tr, X_te = X_train, X_test
        
    # Train
    model.fit(X_tr, y_train)
    
    # Predict
    y_pred_train = model.predict(X_tr)
    y_pred_test = model.predict(X_te)
    
    # Store predictions for CSV
    predictions_dict[name] = y_pred_test
    
    # Calculate Metrics
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    mae = mean_absolute_error(y_test, y_pred_test)
    r2 = r2_score(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    
    elapsed = time.time() - start_time
    
    # Print real-time update
    print(f"{name:<15} | {rmse:<10.4f} | {mae:<10.4f} | {r2:<10.4f} | {elapsed:<10.2f}")
    
    # Append to list for DataFrame
    metrics_list.append({
        'Model': name,
        'Test_RMSE': rmse,
        'Test_MAE': mae,
        'Test_R2': r2,
        'Train_R2': train_r2,
        'Time_Seconds': elapsed
    })
    
    # Store in results dict (structure kept for backward compatibility with your next cells)
    results[name] = {
        'train_rmse': np.sqrt(mean_squared_error(y_train, y_pred_train)),
        'test_rmse': rmse,
        'train_mse': mean_squared_error(y_train, y_pred_train),
        'test_mse': mean_squared_error(y_test, y_pred_test),
        'train_r2': train_r2,
        'test_r2': r2,
        'train_mae': mean_absolute_error(y_train, y_pred_train),
        'test_mae': mae
    }

print("-" * 85)

# ==========================================
# 5. Save Results to CSV (For Visualization)
# ==========================================
# CSV 1: Performance Metrics
metrics_df = pd.DataFrame(metrics_list)
metrics_df.sort_values(by='Test_R2', ascending=False, inplace=True)
metrics_df.to_csv(RESULTS_DIR / 'model_performance_metrics.csv', index=False)
print("\n[Saved] Performance metrics saved to 'model_performance_metrics.csv'")

# CSV 2: Predictions (Actual vs Predicted for all models)
predictions_df = pd.DataFrame(predictions_dict)
predictions_df.to_csv(RESULTS_DIR / 'model_predictions.csv', index=False)
print("[Saved] All model predictions saved to 'model_predictions.csv'")

# Display ranking
print("\n--- Model Ranking (by Test R2) ---")
print(metrics_df[['Model', 'Test_RMSE', 'Test_R2']].to_string(index=False))


In [ ]:
# ==========================================
# CELL 5: Model Performance Evaluation & Selection
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

# --- Visualization Parameters ---
TITLE_SZ = 18
LABEL_SZ = 15
TICK_SZ  = 14
DPI      = 300

# ==========================================
# 1. Load Data from CSV
# ==========================================
# Reading results generated in the previous step
metrics_df = pd.read_csv(RESULTS_DIR / 'model_performance_metrics.csv')
predictions_df = pd.read_csv(RESULTS_DIR / 'model_predictions.csv')

# Calculate MSE (since CSV has RMSE)
metrics_df['Test_MSE'] = metrics_df['Test_RMSE'] ** 2

# Sort by R2 for better visualization ranking
metrics_df = metrics_df.sort_values(by='Test_R2', ascending=True) # Ascending for horizontal bars

print("Data Loaded Successfully.")
print(f"Comparing {len(metrics_df)} models...")

# ==========================================
# 2. Visualization (2x2 Layout)
# ==========================================
print("Generating Performance Comparison Plots...")
fig, axes = plt.subplots(2, 2, figsize=(20, 16))

# Helper function for Horizontal Bar Charts
def plot_hbar(ax, data, metric, color, title, xlabel):
    # Plot bars
    bars = ax.barh(data['Model'], data[metric], color=color, alpha=0.8, edgecolor='black')
    
    # Styling
    ax.set_title(title, fontsize=TITLE_SZ, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=LABEL_SZ, fontweight='bold')
    ax.tick_params(axis='both', labelsize=TICK_SZ)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    
    # Grid (Vertical only)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    
    # Add values at the end of bars
    for bar in bars:
        width = bar.get_width()
        # Adjust text position based on metric scale
        offset = width * 0.01
        ax.text(width + offset, bar.get_y() + bar.get_height()/2, 
                f'{width:.4f}', 
                va='center', ha='left', fontsize=12, fontweight='bold')

# --- Plot 1: RMSE Comparison (Lower is Better) ---
plot_hbar(axes[0,0], metrics_df, 'Test_RMSE', '#FF6B6B', 'RMSE Comparison (Lower is Better)', 'RMSE')

# --- Plot 2: R² Comparison (Higher is Better) ---
plot_hbar(axes[0,1], metrics_df, 'Test_R2', '#4ECDC4', 'R² Comparison (Higher is Better)', 'R-Squared')

# --- Plot 3: MAE Comparison (Lower is Better) ---
plot_hbar(axes[1,0], metrics_df, 'Test_MAE', '#45B7D1', 'MAE Comparison (Lower is Better)', 'MAE')

# --- Plot 4: Prediction vs Actual (Best Model) ---
# Identify Best Model
best_model_name = metrics_df.iloc[-1]['Model'] # Last one because we sorted Ascending
y_actual = predictions_df['Actual']
y_pred_best = predictions_df[best_model_name]

# Scatter Plot
ax4 = axes[1,1]
ax4.scatter(y_actual, y_pred_best, color='royalblue', alpha=0.6, s=80, edgecolors='white')

# Reference Line (Perfect Prediction)
min_val = min(y_actual.min(), y_pred_best.min())
max_val = max(y_actual.max(), y_pred_best.max())
ax4.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Fit')

# Metrics Text
r2_best = metrics_df.iloc[-1]['Test_R2']
rmse_best = metrics_df.iloc[-1]['Test_RMSE']
text_str = f'Model: {best_model_name}\n$R^2$ = {r2_best:.4f}\nRMSE = {rmse_best:.4f}'
ax4.text(0.05, 0.95, text_str, transform=ax4.transAxes, fontsize=14, fontweight='bold',
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

ax4.set_title(f'Best Model Performance: {best_model_name}', fontsize=TITLE_SZ, fontweight='bold')
ax4.set_xlabel('Actual Values', fontsize=LABEL_SZ, fontweight='bold')
ax4.set_ylabel('Predicted Values', fontsize=LABEL_SZ, fontweight='bold')
ax4.tick_params(axis='both', labelsize=TICK_SZ)
for label in ax4.get_xticklabels() + ax4.get_yticklabels():
    label.set_fontweight('bold')
ax4.grid(True, linestyle='--', alpha=0.5)
ax4.legend(fontsize=12, loc='lower right')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'model_comparison_summary.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 3. Model Selection & Export
# ==========================================
# Select Top 2 Models for further optimization
# Sort Descending to get top models first
metrics_df_desc = metrics_df.sort_values(by='Test_R2', ascending=False)

print("\n--- Final Model Ranking (Sorted by R²) ---")
print(metrics_df_desc[['Model', 'Test_R2', 'Test_RMSE', 'Test_MAE']].to_string(index=False))

best_models = metrics_df_desc.head(2)['Model'].tolist()
print(f"\n[Decision] Top 2 Models selected for optimization: {best_models}")

# Save selection to a small CSV for record
selection_df = pd.DataFrame({'Top_Models': best_models})
selection_df.to_csv(RESULTS_DIR / 'top_models_selected.csv', index=False)
print("Saved selection to 'top_models_selected.csv'")


In [ ]:
# ==========================================
# CELL 6: Data Split Sensitivity Analysis (Final Range: 20-35%)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings

warnings.filterwarnings("default")

# --- Visualization Parameters ---
TITLE_SZ = 18
LABEL_SZ = 14
TICK_SZ  = 13
DPI      = 300

# ==========================================
# 1. Setup & Configuration
# ==========================================
# Load the top models selected in the previous step.
top_models_path = RESULTS_DIR / "top_models_selected.csv"
if not top_models_path.exists():
    raise RuntimeError("Run the model evaluation and selection cell before this cell.")
top_models_df = pd.read_csv(top_models_path)
target_models = top_models_df["Top_Models"].tolist()
print(f"Performing sensitivity analysis on top models: {target_models}")

# FINAL ADJUSTED RANGE: 20%, 25%, 30%, 35%
test_sizes = [0.20, 0.25, 0.30, 0.35]
print(f"Testing split ratios: {test_sizes}")

def get_model_instance(name):
    if name == 'RandomForest': return RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED)
    if name == 'CatBoost': return CatBoostRegressor(iterations=100, random_state=RANDOM_SEED, verbose=0, allow_writing_files=False)
    if name == 'XGBoost': return xgb.XGBRegressor(n_estimators=100, random_state=RANDOM_SEED, eval_metric='rmse')
    if name == 'LightGBM': return lgb.LGBMRegressor(n_estimators=100, random_state=RANDOM_SEED, verbose=-1)
    if name == 'GBR': return GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_SEED)
    if name == 'SVR': return SVR(kernel='rbf', C=100, gamma='scale')
    if name == 'ElasticNet': return ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=RANDOM_SEED)
    return None

# ==========================================
# 2. Iterative Testing Loop
# ==========================================
sensitivity_data = []

print("\nStarting Sensitivity Loop...")
for size in test_sizes:
    # 1. Split Data
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=size, random_state=RANDOM_SEED)
    
    # 2. Preprocessing
    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)
    
    for model_name in target_models:
        model = get_model_instance(model_name)
        
        # Select Data Type
        if model_name in ['SVR', 'ElasticNet']:
            X_train_curr, X_test_curr = X_tr_sc, X_te_sc
        else:
            X_train_curr, X_test_curr = X_tr, X_te
            
        # Train & Predict
        model.fit(X_train_curr, y_tr)
        y_pred = model.predict(X_test_curr)
        
        # Calculate Metrics
        mse = mean_squared_error(y_te, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_te, y_pred)
        
        sensitivity_data.append({
            'Test_Pct': int(size * 100), # Store as percentage (20, 25, etc.)
            'Model': model_name,
            'RMSE': rmse,
            'MSE': mse,
            'R2': r2
        })

# ==========================================
# 3. Save Results
# ==========================================
results_df = pd.DataFrame(sensitivity_data)
results_df.to_csv(RESULTS_DIR / 'split_sensitivity_results.csv', index=False)
print("\n[Saved] Results saved to 'split_sensitivity_results.csv'")

# Find Best Configuration (Maximize R2)
best_run = results_df.sort_values(by='R2', ascending=False).iloc[0]
best_split_pct = best_run['Test_Pct']
best_model_final = best_run['Model']

print(f"\n[CONCLUSION] Best Configuration found:")
print(f"   Model: {best_model_final}")
print(f"   Split: {100-best_split_pct}/{best_split_pct}")
print(f"   R2:    {best_run['R2']:.4f}")

# Save best config
with open(RESULTS_DIR / 'best_config_info.txt', 'w') as f:
    f.write(f"{best_model_final},{best_split_pct/100}")

# ==========================================
# 4. Visualization (Three Column Style)
# ==========================================
print("Generating 3-Column Sensitivity Plots...")

# Metrics to plot
metrics_to_plot = ['RMSE', 'MSE', 'R2']
titles = ['RMSE', 'MSE', 'R2'] # Short clean titles
colors = ['steelblue', 'orange', 'green'] # Custom colors for models

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

# Get unique models for consistent coloring
unique_models = results_df['Model'].unique()
model_colors = {model: color for model, color in zip(unique_models, ['steelblue', 'orange', 'green'])}
markers = ['o', 's', '^'] # Different markers

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    
    # Plot each model's line
    for j, model in enumerate(unique_models):
        subset = results_df[results_df['Model'] == model]
        
        ax.plot(subset['Test_Pct'], subset[metric], 
                marker=markers[j % len(markers)], 
                markersize=10, 
                linewidth=2.5,
                label=model,
                color=model_colors[model])
        
        # Add Value Labels (Text above points)
        for _, row in subset.iterrows():
            val = row[metric]
            # Offset text slightly to avoid overlap with line
            offset_y = (results_df[metric].max() - results_df[metric].min()) * 0.05
            
            # Text formatting
            ax.text(row['Test_Pct'], val + offset_y, 
                    f'{val:.3f}', 
                    ha='center', va='bottom', 
                    fontsize=12, fontweight='bold', color='black')

    ax.set_title(titles[i], fontsize=TITLE_SZ, fontweight='bold')
    ax.set_xlabel('Test Set Percentage (%)', fontsize=LABEL_SZ, fontweight='bold')
    ax.set_ylabel(metric, fontsize=LABEL_SZ, fontweight='bold')
    ax.set_xticks(test_sizes_pct := [int(x*100) for x in test_sizes]) # Explicit ticks [20, 25, 30, 35]
    ax.tick_params(labelsize=TICK_SZ)
    
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight('bold')
    
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.legend(fontsize=12)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'split_sensitivity_3col.png', dpi=DPI, bbox_inches='tight')
plt.show()


In [ ]:
# ==========================================
# CELL 7: Hyperparameter Optimization (Final Flexible Version)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from matplotlib.patches import Rectangle
import matplotlib.lines as mlines
import warnings

warnings.filterwarnings("default")

# ==========================================
# --- Font size control ---
# Adjust these values to control plot text sizes.
# ==========================================
DPI = 300
FONT_FAMILY = 'Arial'

LABEL_SZ  = 16    # Axis labels (for example, "Actual Values")
TICK_SZ   = 14    # Tick labels (for example, "0, 20, 40")
VAL_SZ    = 11    # Numeric annotations in plots and heatmaps
LEGEND_SZ = 12    # Legend text

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.5 

# ==========================================
# 1. Configuration & Data Split
# ==========================================
best_config_path = RESULTS_DIR / "best_config_info.txt"
if not best_config_path.exists():
    raise RuntimeError("Run the data-split sensitivity analysis cell before this cell.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_model_name = content[0]
best_split_ratio = float(content[1])
print(f"Target Model: {best_model_name}")
print(f"Split Ratio: {100-best_split_ratio*100:.0f}/{best_split_ratio*100:.0f}")

# Split Data
X_train_opt, X_test_opt, y_train_opt, y_test_opt = train_test_split(
    X, y, test_size=best_split_ratio, random_state=RANDOM_SEED
)

# Preprocessing
if best_model_name in ['SVR', 'ElasticNet']:
    scaler = StandardScaler()
    X_train_final = scaler.fit_transform(X_train_opt)
    X_test_final = scaler.transform(X_test_opt)
else:
    X_train_final = X_train_opt
    X_test_final = X_test_opt

# ==========================================
# 2. Grid Search Execution
# ==========================================
param_grids = {
    'RandomForest': {
        'n_estimators': [50, 100, 200, 300, 500, 800],
        'max_depth': [10, 15, 20, 25, 30, 35, 40]
    },
    'CatBoost': {
        'iterations': [500, 800, 1000, 1500, 2000],
        'depth': [4, 6, 8, 10],
    }
}
current_grid = param_grids.get(best_model_name, param_grids['RandomForest'])

def get_base_model(name):
    if name == 'RandomForest': return RandomForestRegressor(random_state=RANDOM_SEED, n_jobs=-1)
    if name == 'CatBoost': return CatBoostRegressor(random_state=RANDOM_SEED, verbose=0, allow_writing_files=False)
    return RandomForestRegressor(random_state=RANDOM_SEED)

model = get_base_model(best_model_name)

print(f"\nRunning Grid Search for {best_model_name}...")
grid_search = GridSearchCV(
    estimator=model,
    param_grid=current_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train_final, y_train_opt)

best_params = grid_search.best_params_
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results['mean_test_rmse'] = -cv_results['mean_test_score']

print(f"Best Params: {best_params}")

with (RESULTS_DIR / "best_hyperparams.json").open("w", encoding="utf-8") as f:
    json.dump(best_params, f, indent=2)
cv_results.to_csv(RESULTS_DIR / "hyperparameter_cv_results.csv", index=False)
print("Optimized hyperparameters and cross-validation results saved in the results directory.")

# ==========================================
# 3. Data Preparation for Plotting
# ==========================================
params = list(current_grid.keys())
p_x = params[0] # e.g. n_estimators
p_y = params[1] # e.g. max_depth
p_x_col = f'param_{p_x}'
p_y_col = f'param_{p_y}'

# Prepare Trend Data
cv_results[p_y_col] = cv_results[p_y_col].fillna('None')
grouped = cv_results.groupby(p_y_col)['mean_test_rmse'].min()
# Sort logic
try:
    idx_numeric = sorted([x for x in grouped.index if str(x) != 'None'], key=lambda x: int(x))
    new_index = idx_numeric + ['None'] if 'None' in grouped.index else idx_numeric
    grouped = grouped.reindex(new_index)
except (TypeError, ValueError):
    pass
x_vals = [str(x) for x in grouped.index]
y_vals = grouped.values

# Prepare Heatmap Data
pivot_table = cv_results.pivot(index=p_y_col, columns=p_x_col, values='mean_test_rmse')
try:
    pivot_table.index = pivot_table.index.astype(int)
    pivot_table.sort_index(inplace=True)
except (TypeError, ValueError):
    pass

# Prepare Scatter Data (Retrain)
best_model_final = grid_search.best_estimator_
y_pred_tr = best_model_final.predict(X_train_final)
y_pred_te = best_model_final.predict(X_test_final)
r2_tr = r2_score(y_train_opt, y_pred_tr)
rmse_tr = np.sqrt(mean_squared_error(y_train_opt, y_pred_tr))
r2_te = r2_score(y_test_opt, y_pred_te)
rmse_te = np.sqrt(mean_squared_error(y_test_opt, y_pred_te))

# ==========================================
# 4. Plotting Functions (Reusable)
# ==========================================
def plot_trend(ax):
    # Colors: Deep Sky Blue Bars (#2b7bba), Navy Line (#000080)
    ax.bar(x_vals, y_vals, color='#2b7bba', edgecolor='black', alpha=1.0, width=0.6)
    ax.plot(x_vals, y_vals, color='#000080', marker='o', linestyle='-', linewidth=3, markersize=8)
    
    ax.set_xlabel(p_y, fontsize=LABEL_SZ, fontweight='bold')
    ax.set_ylabel('RMSE', fontsize=LABEL_SZ, fontweight='bold')
    ax.tick_params(labelsize=TICK_SZ)
    
    ymin, ymax = min(y_vals)*0.98, max(y_vals)*1.02
    ax.set_ylim(ymin, ymax)
    
    for i, v in enumerate(y_vals):
        ax.text(i, v + (ymax-ymin)*0.02, f'{v:.2f}', ha='center', fontweight='bold', fontsize=VAL_SZ)

def plot_heatmap(ax):
    sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='viridis_r', ax=ax,
                cbar_kws={'label': 'RMSE'}, linewidths=1, linecolor='white',
                annot_kws={'size': VAL_SZ, 'weight': 'bold'}) # Use VAL_SZ here
    
    # Highlight Best
    best_py_val = best_params[p_y]
    best_px_val = best_params[p_x]
    if best_py_val is None: best_py_val = 'None'
    try:
        r_idx = list(pivot_table.index).index(best_py_val)
        c_idx = list(pivot_table.columns).index(best_px_val)
        ax.add_patch(Rectangle((c_idx, r_idx), 1, 1, fill=False, edgecolor='red', lw=4, clip_on=False))
    except (KeyError, TypeError, ValueError):
        pass
    
    ax.set_xlabel(p_x, fontsize=LABEL_SZ, fontweight='bold')
    ax.set_ylabel(p_y, fontsize=LABEL_SZ, fontweight='bold')
    ax.tick_params(labelsize=TICK_SZ)

def plot_scatter(ax):
    min_val = min(min(y_train_opt), min(y_test_opt)) * 0.95
    max_val = max(max(y_train_opt), max(y_test_opt)) * 1.05
    
    ax.scatter(y_train_opt, y_pred_tr, c='#4169E1', marker='^', alpha=0.6, s=70)
    ax.scatter(y_test_opt, y_pred_te, c='#D62728', marker='o', alpha=0.7, s=70)
    ax.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=2)
    
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    ax.set_aspect('equal', adjustable='box')
    
    ax.set_xlabel('Actual Values', fontsize=LABEL_SZ, fontweight='bold')
    ax.set_ylabel('Predicted Values', fontsize=LABEL_SZ, fontweight='bold')
    ax.tick_params(labelsize=TICK_SZ)
    ax.grid(True, linestyle='--', alpha=0.3)
    
    # Legend with Metrics
    blue_tri = mlines.Line2D([], [], color='#4169E1', marker='^', linestyle='None', markersize=10, 
                             label=f"Train $R^2$={r2_tr:.2f}\nTrain RMSE={rmse_tr:.2f}")
    red_circ = mlines.Line2D([], [], color='#D62728', marker='o', linestyle='None', markersize=10, 
                             label=f"Test $R^2$={r2_te:.2f}\nTest RMSE={rmse_te:.2f}")
    ax.legend(handles=[blue_tri, red_circ], loc='upper left', frameon=True, fontsize=LEGEND_SZ)

# ==========================================
# 5. Generate Output 1: Combined Image
# ==========================================
print("\nGenerating Combined Image...")
fig, axes = plt.subplots(1, 3, figsize=(24, 7.5))
plot_trend(axes[0])
plot_heatmap(axes[1])
plot_scatter(axes[2])
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'Hyperparam_Opt_Combined.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 6. Generate Output 2: Separate Images
# ==========================================
print("\nGenerating Separate Images...")

# Image A
fig_a = plt.figure(figsize=(8, 6))
ax_a = plt.gca()
plot_trend(ax_a)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'Hyperparam_Trend.png', dpi=DPI, bbox_inches='tight')
plt.close()

# Image B
fig_b = plt.figure(figsize=(9, 7)) # Slightly wider for colorbar
ax_b = plt.gca()
plot_heatmap(ax_b)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'Hyperparam_Heatmap.png', dpi=DPI, bbox_inches='tight')
plt.close()

# Image C
fig_c = plt.figure(figsize=(8, 8)) # Square
ax_c = plt.gca()
plot_scatter(ax_c)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'Hyperparam_Scatter.png', dpi=DPI, bbox_inches='tight')
plt.close()

print("All images saved successfully.")


In [ ]:
# ==========================================
# CELL 8: Final Evaluation & Feature Importance (SCI Boxed Style)
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import json
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import matplotlib.lines as mlines
from matplotlib.ticker import MaxNLocator
import warnings

warnings.filterwarnings("default")

# ==========================================
# --- Publication-style visualization settings ---
# ==========================================
DPI = 300
FONT_FAMILY = 'Arial'
LABEL_SZ  = 16    
TICK_SZ   = 14    
VAL_SZ    = 11    
LEGEND_SZ = 12    

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.5 # Thicker border
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5
plt.rcParams['xtick.direction'] = 'in' # Ticks inside
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.grid'] = True   # Turn grid on
plt.rcParams['grid.alpha'] = 0.3   # Light grid
plt.rcParams['grid.linestyle'] = '--'

# ==========================================
# 1. Load & Train
# ==========================================
print("1. Training Optimal Model...")
best_config_path = RESULTS_DIR / "best_config_info.txt"
best_params_path = RESULTS_DIR / "best_hyperparams.json"
if not best_config_path.exists() or not best_params_path.exists():
    raise RuntimeError("Run the sensitivity and hyperparameter optimization cells first.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_model_name = content[0]
best_split_ratio = float(content[1])
with best_params_path.open("r", encoding="utf-8") as f:
    best_params = json.load(f)
if best_model_name != "RandomForest":
    raise ValueError(
        "The downstream interpretation cells currently support RandomForest only; "
        f"the selected model was {best_model_name}."
    )

# Data Split
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=best_split_ratio, random_state=RANDOM_SEED
)

# Train RF
model = RandomForestRegressor(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train_final, y_train_final)

# Predictions
y_pred_train = model.predict(X_train_final)
y_pred_test = model.predict(X_test_final)
residuals_test = y_test_final - y_pred_test

# Metrics
r2_tr = r2_score(y_train_final, y_pred_train)
r2_te = r2_score(y_test_final, y_pred_test)
rmse_te = np.sqrt(mean_squared_error(y_test_final, y_pred_test))

# ==========================================
# 2. Residual Analysis (1x3 Boxed)
# ==========================================
print("2. Generating Residual Plots...")
fig, axes = plt.subplots(1, 3, figsize=(26, 8))

# --- Left: Actual vs Predicted ---
min_val = min(y.min(), y_pred_test.min()) * 0.95
max_val = max(y.max(), y_pred_test.max()) * 1.05

axes[0].scatter(y_train_final, y_pred_train, c='#4169E1', marker='^', alpha=0.6, s=70, label='Train')
axes[0].scatter(y_test_final, y_pred_test, c='#D62728', marker='o', alpha=0.7, s=70, label='Test')
axes[0].plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1.5)

axes[0].set_xlim(min_val, max_val)
axes[0].set_ylim(min_val, max_val)
axes[0].set_aspect('equal', adjustable='box')
axes[0].set_xlabel('Actual Values', fontsize=LABEL_SZ, fontweight='bold')
axes[0].set_ylabel('Predicted Values', fontsize=LABEL_SZ, fontweight='bold')
axes[0].tick_params(labelsize=TICK_SZ)

blue_tri = mlines.Line2D([], [], color='#4169E1', marker='^', linestyle='None', markersize=10, 
                         label=f"Train $R^2$={r2_tr:.2f}")
red_circ = mlines.Line2D([], [], color='#D62728', marker='o', linestyle='None', markersize=10, 
                         label=f"Test $R^2$={r2_te:.2f}\nRMSE={rmse_te:.2f}")
axes[0].legend(handles=[blue_tri, red_circ], loc='upper left', frameon=True, fontsize=LEGEND_SZ, fancybox=False, edgecolor='black')

# --- Middle: Residual Scatter ---
axes[1].scatter(y_test_final, residuals_test, c='#2878B5', alpha=0.7, s=70, edgecolor='black', linewidth=0.5)
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Actual Values', fontsize=LABEL_SZ, fontweight='bold')
axes[1].set_ylabel('Residuals', fontsize=LABEL_SZ, fontweight='bold')
axes[1].tick_params(labelsize=TICK_SZ)

# --- Right: Residual Histogram ---
sns.histplot(residuals_test, kde=True, ax=axes[2], color='#9AC9DB', edgecolor='black')
axes[2].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[2].set_xlabel('Residual Error', fontsize=LABEL_SZ, fontweight='bold')
axes[2].set_ylabel('Frequency', fontsize=LABEL_SZ, fontweight='bold')
axes[2].tick_params(labelsize=TICK_SZ)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'Final_Residual_Analysis.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 3. Feature Importance Preparation
# ==========================================
importances = model.feature_importances_
fi_df = pd.DataFrame({'Feature': X.columns, 'Importance': importances})
fi_df['Feature'] = fi_df['Feature'].apply(lambda x: x.replace('℃', '°C'))
fi_df = fi_df.sort_values('Importance', ascending=True) # For Horizontal Bar
fi_df_desc = fi_df.sort_values('Importance', ascending=False) # For Vertical/Cumulative

# Save CSV
fi_df_desc.to_csv(RESULTS_DIR / 'final_feature_importance.csv', index=False)

# ==========================================
# 4. Visualization B: Feature Importance (Refined)
# ==========================================

# --- Plot 1: All Features (Ref Image 2 Left Style - Light Blue Fill, Dark Edge) ---
plt.figure(figsize=(12, 14))
ax = plt.gca()

# Light Blue fill, Dark Blue edge
bars = ax.barh(fi_df['Feature'], fi_df['Importance'], color='#B3E5FC', edgecolor='#01579B', linewidth=1.2, height=0.7)

ax.set_xlabel('Feature Importance', fontsize=LABEL_SZ, fontweight='bold')
ax.tick_params(labelsize=TICK_SZ)
ax.set_xlim(0, max(fi_df['Importance']) * 1.15) # Space for labels

# Add Values
for bar in bars:
    width = bar.get_width()
    ax.text(width + 0.001, bar.get_y() + bar.get_height()/2, 
             f'{width:.3f}', va='center', fontsize=11, fontweight='bold', color='black')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'FI_All_Features.png', dpi=DPI)
plt.show()


# --- Plot 2: Top 16 Features (Ref Image 2 Right Style - Plasma Gradient) ---
top_16 = fi_df.tail(16).sort_values('Importance', ascending=False) # Descending for vertical plot

plt.figure(figsize=(14, 8))
ax = plt.gca()

# Gradient Colors (Plasma: Purple -> Yellow)
cmap = plt.get_cmap('plasma')
norm = mcolors.Normalize(vmin=top_16['Importance'].min(), vmax=top_16['Importance'].max())
colors_16 = [cmap(norm(v)) for v in top_16['Importance']]

bars = ax.bar(top_16['Feature'], top_16['Importance'], color=colors_16, edgecolor='black', width=0.7)

ax.set_ylabel('Feature Importance', fontsize=LABEL_SZ, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=TICK_SZ, fontweight='bold')
plt.yticks(fontsize=TICK_SZ)
ax.set_ylim(0, max(top_16['Importance']) * 1.1)

# Add Values on Top
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height + 0.002, 
             f'{height:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'FI_Top16_Plasma.png', dpi=DPI)
plt.show()


# --- Plot 3: Cumulative Importance (Ref Image 2 Bottom Right Style) ---
cumulative_imp = np.cumsum(fi_df_desc['Importance'].values)
if cumulative_imp[-1] > 1.1: cumulative_imp /= cumulative_imp[-1] # Normalize if needed

plt.figure(figsize=(10, 7))
ax = plt.gca()

# Line Plot
ax.plot(range(1, len(cumulative_imp)+1), cumulative_imp, 
        color='#8B0000', marker='o', linewidth=3, markersize=7, label='Cumulative Importance')

# Threshold Lines
ax.axhline(y=0.8, color='green', linestyle='--', linewidth=2, label='80% Threshold')
ax.axhline(y=0.9, color='orange', linestyle='--', linewidth=2, label='90% Threshold')

# Find crossing points for annotation
idx_80 = np.argmax(cumulative_imp >= 0.8) + 1
idx_90 = np.argmax(cumulative_imp >= 0.9) + 1

# Vertical drop lines
ax.axvline(x=idx_80, ymax=0.8, color='green', linestyle=':', alpha=0.7)
ax.axvline(x=idx_90, ymax=0.9, color='orange', linestyle=':', alpha=0.7)

# Annotations (Box style like reference)
ax.text(idx_80, 0.82, f'{idx_80} features\n(80%)', ha='center', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.6))
ax.text(idx_90, 0.92, f'{idx_90} features\n(90%)', ha='center', fontsize=10, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='navajowhite', alpha=0.6))

ax.set_xlabel('Number of Features', fontsize=LABEL_SZ, fontweight='bold')
ax.set_ylabel('Cumulative Importance', fontsize=LABEL_SZ, fontweight='bold')
ax.tick_params(labelsize=TICK_SZ)
ax.legend(fontsize=LEGEND_SZ, loc='lower right', frameon=True, edgecolor='black')
# Set X-axis to integers
ax.xaxis.set_major_locator(MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'FI_Cumulative_Analysis.png', dpi=DPI)
plt.show()


# --- Plot 4: Top 9 Contribution (Pie - Exploded & Colorful) ---
top_9 = fi_df_desc.head(9)
others_sum = fi_df_desc.iloc[9:]['Importance'].sum()
pie_data = list(top_9['Importance']) + [others_sum]
pie_labels = list(top_9['Feature']) + ['Others']

# Vibrant Colors
pie_colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEEAD', 
              '#FFD93D', '#FF9F68', '#A7226E', '#2F4F4F', '#D3D3D3']
explode = [0.05] * len(pie_data) 

plt.figure(figsize=(9, 9))
wedges, texts, autotexts = plt.pie(pie_data, labels=pie_labels, autopct='%1.1f%%', 
                                   startangle=90, colors=pie_colors, explode=explode,
                                   shadow=True, counterclock=False,
                                   wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
                                   textprops={'fontsize': 13, 'fontweight': 'bold'})

for text in texts: text.set_color('black')
for autotext in autotexts: autotext.set_color('white')

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'FI_Top9_Pie.png', dpi=DPI)
plt.show()

print("\nVisualizations Generated (Boxed Style):")
print("1. Final_Residual_Analysis.png")
print("2. FI_All_Features.png (Light Blue Bars)")
print("3. FI_Top16_Plasma.png (Gradient Vertical)")
print("4. FI_Cumulative_Analysis.png (Line with Thresholds)")
print("5. FI_Top9_Pie.png")


In [ ]:
# ==========================================
# CELL 9: SHAP Analysis (Fixed & Minimalist)
# ==========================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import shap
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from matplotlib.colors import LinearSegmentedColormap
import warnings

warnings.filterwarnings("default")

# --- Visualization settings ---
DPI = 300
FONT_FAMILY = 'Arial'
LABEL_SZ = 16    
TICK_SZ = 14     
VAL_SZ = 11

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.5 
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

# Custom Vivid Colormap (Blue -> Red)
colors = ["#008bfb", "#ff0051"] 
shap_cmap = LinearSegmentedColormap.from_list("shap_vivid", colors)

# ==========================================
# 1. Load Config & Retrain
# ==========================================
print("1. Loading & Retraining...")
best_config_path = RESULTS_DIR / "best_config_info.txt"
best_params_path = RESULTS_DIR / "best_hyperparams.json"
if not best_config_path.exists() or not best_params_path.exists():
    raise RuntimeError("Run the sensitivity and hyperparameter optimization cells first.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_split_ratio = float(content[1])
with best_params_path.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=best_split_ratio, random_state=RANDOM_SEED
)

# Fix Feature Names
X_test_display = X_test.copy()
new_cols = {col: col.replace('℃', '°C') for col in X_test_display.columns}
X_test_display.rename(columns=new_cols, inplace=True)

model = RandomForestRegressor(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train, y_train)

# ==========================================
# 2. Calculate SHAP
# ==========================================
print("2. Calculating SHAP Values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# ==========================================
# 3. Plot 1: SHAP Beeswarm (Minimalist)
# ==========================================
print("3. Generating SHAP Beeswarm (No Grid, Solid Axis)...")

plt.figure(figsize=(12, 10))

# Draw SHAP plot
shap.summary_plot(
    shap_values, 
    X_test_display, 
    plot_type="dot", 
    max_display=16, 
    show=False,
    cmap=shap_cmap, 
    alpha=1.0, 
    plot_size=(12, 10)
)

ax = plt.gca()

# --- SCI STYLING ---
# 1. Remove all grid lines
ax.grid(False)

# 2. Add ONLY the zero line (Vertical)
ax.axvline(x=0, color="#888888", linestyle="-", linewidth=1.0, zorder=-1)

# 3. Solid Black X-Axis (Bottom Spine)
ax.spines['bottom'].set_visible(True)
ax.spines['bottom'].set_linewidth(1.5)
ax.spines['bottom'].set_color('black')

# 4. Hide Top/Right/Left spines
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_visible(False)

# 5. Bold Labels
ax.set_xlabel('SHAP value (impact on model output)', fontsize=LABEL_SZ, fontweight='bold')
plt.yticks(fontsize=TICK_SZ, fontweight='bold')
plt.xticks(fontsize=TICK_SZ, fontweight='bold')

# 6. Colorbar Adjustments (FIXED)
f = plt.gcf()
if len(f.get_axes()) > 1:
    cbar_ax = f.get_axes()[-1]
    cbar_ax.tick_params(labelsize=12)
    cbar_ax.set_ylabel('Feature value', fontsize=16, fontweight='bold')
    
    # FIX: Remove outline using spines instead of .outline attribute
    for spine in cbar_ax.spines.values():
        spine.set_visible(False)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'SHAP_Summary_Beeswarm.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 4. Plot 2: SHAP Bar (Boxed, No X-Ticks)
# ==========================================
print("4. Generating SHAP Bar Chart (Clean)...")

# Data Prep
mean_abs_shap = np.abs(shap_values).mean(axis=0)
shap_importance = pd.DataFrame({
    'Feature': X_test_display.columns,
    'Mean_SHAP': mean_abs_shap
}).sort_values(by='Mean_SHAP', ascending=True)

# Dynamic height
n_features = len(shap_importance)
fig_height = max(10, n_features * 0.4)

plt.figure(figsize=(12, fig_height))
ax = plt.gca()

# EST Blue
EST_BLUE = '#2878B5'

# Bars
bars = ax.barh(shap_importance['Feature'], shap_importance['Mean_SHAP'], 
               color=EST_BLUE, edgecolor='black', linewidth=1.2, height=0.7)

# Add Values
max_val = shap_importance['Mean_SHAP'].max()
for bar in bars:
    width = bar.get_width()
    ax.text(width + max_val*0.01, bar.get_y() + bar.get_height()/2, 
            f'{width:.3f}', 
            va='center', fontsize=VAL_SZ, fontweight='bold', color='black')

# --- CLEAN STYLE ---
# 1. Concise Label
ax.set_xlabel('mean(|SHAP value|)', fontsize=LABEL_SZ, fontweight='bold')

# 2. Y Ticks only
plt.yticks(fontsize=TICK_SZ, fontweight='bold')

# 3. REMOVE X Ticks and Labels
plt.xticks([]) 

# 4. Boxed Spines (Full Border)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.5)
    spine.set_color('black')

# 5. Remove Grid
ax.grid(False)

ax.set_xlim(0, max_val * 1.15)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'SHAP_Global_Importance_Bar.png', dpi=DPI, bbox_inches='tight')
plt.show()

print("\nSHAP Plots Generated.")


In [ ]:
# ==========================================
# CELL 10: Partial Dependence Plots (Fixed for Sklearn Version)
# ==========================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence
import warnings

warnings.filterwarnings("default")

# --- Visualization settings ---
DPI = 300
FONT_FAMILY = 'Arial'
LABEL_SZ = 16
TICK_SZ = 14
LINE_WIDTH = 2.5
LINE_COLOR = '#0000FF' # Pure Blue

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.5 
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5

# ==========================================
# 1. Load Config & Retrain
# ==========================================
print("1. Loading Config...")
best_config_path = RESULTS_DIR / "best_config_info.txt"
best_params_path = RESULTS_DIR / "best_hyperparams.json"
if not best_config_path.exists() or not best_params_path.exists():
    raise RuntimeError("Run the sensitivity and hyperparameter optimization cells first.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_split_ratio = float(content[1])
with best_params_path.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=best_split_ratio, random_state=RANDOM_SEED
)

# Fix Feature Names
X_train_display = X_train.copy()
new_cols = {col: col.replace('℃', '°C') for col in X_train_display.columns}
X_train_display.rename(columns=new_cols, inplace=True)

model = RandomForestRegressor(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train_display, y_train)

# ==========================================
# 2. Get Top 9 Features
# ==========================================
feature_importance_path = RESULTS_DIR / "final_feature_importance.csv"
if not feature_importance_path.exists():
    raise RuntimeError("Run the final evaluation and feature-importance cell first.")
fi_df = pd.read_csv(feature_importance_path)
fi_df["Feature"] = fi_df["Feature"].str.replace("℃", "°C", regex=False)
top_9_features = fi_df["Feature"].head(9).tolist()

print(f"Generating PDPs for: {top_9_features}")

# ==========================================
# 3. Helper Function for Data Extraction
# ==========================================
def get_pdp_data(model, X, feature):
    """Robust function to handle different sklearn versions"""
    pdp_results = partial_dependence(
        model, X, [feature], kind='average', grid_resolution=50
    )
    
    # Try 'grid_values' (New sklearn), fall back to 'values' (Old sklearn)
    if 'grid_values' in pdp_results:
        x_vals = pdp_results['grid_values'][0]
    elif 'values' in pdp_results:
        x_vals = pdp_results['values'][0]
    else:
        raise KeyError("Could not find 'grid_values' or 'values' in PDP result.")
        
    y_vals = pdp_results['average'][0]
    return x_vals, y_vals

# ==========================================
# 4. Plot A: Combined 3x3 (EST Style)
# ==========================================
print("\n2. Drawing Combined PDP (EST Style)...")

fig, axes = plt.subplots(3, 3, figsize=(18, 15))
axes = axes.flatten()

for i, feature in enumerate(top_9_features):
    ax = axes[i]
    
    # 1. Get Data
    x_vals, y_vals = get_pdp_data(model, X_train_display, feature)
    
    # 2. Plot Line
    ax.plot(x_vals, y_vals, color=LINE_COLOR, linewidth=LINE_WIDTH)
    
    # 3. Find Elbow & Annotate
    dy = np.diff(y_vals)
    if len(dy) > 0:
        max_change_idx = np.argmax(np.abs(dy))
        elbow_x = x_vals[max_change_idx]
        
        # Only annotate if change is significant
        if (max(y_vals) - min(y_vals)) > (np.mean(y_vals) * 0.02):
            ax.axvline(x=elbow_x, color='red', linestyle='-.', linewidth=1.2, ymax=0.9)
            # Smart text positioning
            text_y = min(y_vals) + (max(y_vals)-min(y_vals))*0.1
            ax.text(elbow_x, text_y, f'{elbow_x:.1f}', color='red', 
                    fontweight='bold', fontsize=12, ha='left')

    # 4. Styling
    ax.set_xlabel(feature, fontsize=LABEL_SZ, fontweight='bold')
    if i % 3 == 0:
        ax.set_ylabel("Predicted RHg (%)", fontsize=LABEL_SZ, fontweight='bold')
    
    ax.tick_params(axis='both', which='major', labelsize=TICK_SZ)
    
    # Y-limit scaling
    y_range = max(y_vals) - min(y_vals)
    if y_range < 1.0:
        ax.set_ylim(min(y_vals)-1, max(y_vals)+1)
    else:
        padding = y_range * 0.1
        ax.set_ylim(min(y_vals)-padding, max(y_vals)+padding)

    ax.grid(False)

plt.tight_layout()
plt.subplots_adjust(wspace=0.25, hspace=0.35)
plt.savefig(RESULTS_DIR / 'PDP_Combined_EST_Style.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 5. Plot B: Individual Plots
# ==========================================
print("\n3. Saving Single Plots...")

for feature in top_9_features:
    fig, ax = plt.subplots(figsize=(6, 5))
    
    x_vals, y_vals = get_pdp_data(model, X_train_display, feature)
    
    ax.plot(x_vals, y_vals, color=LINE_COLOR, linewidth=3)
    ax.set_xlabel(feature, fontsize=18, fontweight='bold')
    ax.set_ylabel("Predicted RHg (%)", fontsize=18, fontweight='bold')
    ax.tick_params(labelsize=15)
    ax.grid(False)
    
    # Annotation
    dy = np.diff(y_vals)
    if len(dy) > 0 and (max(y_vals) - min(y_vals)) > 0.1:
        max_change_idx = np.argmax(np.abs(dy))
        elbow_x = x_vals[max_change_idx]
        ax.axvline(x=elbow_x, color='red', linestyle='-.', linewidth=1.5, ymax=0.9)
        ax.text(elbow_x, min(y_vals), f'{elbow_x:.1f}', color='red', 
                fontweight='bold', fontsize=14, ha='left', va='bottom')

    safe_name = feature.replace('(', '').replace(')', '').replace('°C', 'C').replace('/', '_')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'PDP_Single_{safe_name}.png', dpi=DPI, bbox_inches='tight')
    plt.close()

print("All PDP plots generated successfully.")


In [ ]:
# ==========================================
# CELL 11: 3D Partial Dependence Plots (Combined & All Singles)
# ==========================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
import warnings

warnings.filterwarnings("default")

# --- Visualization settings ---
DPI = 300
FONT_FAMILY = 'Arial'
LABEL_SZ = 12
TICK_SZ = 10

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.2

# ==========================================
# 1. Load Config & Retrain
# ==========================================
print("1. Loading Config & Retraining Model...")
best_config_path = RESULTS_DIR / "best_config_info.txt"
best_params_path = RESULTS_DIR / "best_hyperparams.json"
if not best_config_path.exists() or not best_params_path.exists():
    raise RuntimeError("Run the sensitivity and hyperparameter optimization cells first.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_split_ratio = float(content[1])
with best_params_path.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=best_split_ratio, random_state=RANDOM_SEED
)

# Fix Feature Names (For Display: ensure consistent °C)
X_train_display = X_train.copy()
new_cols = {col: col.replace('℃', '°C') for col in X_train_display.columns}
X_train_display.rename(columns=new_cols, inplace=True)

model = RandomForestRegressor(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train_display, y_train)

# ==========================================
# 2. Define Feature Pairs
# ==========================================
# List of all requested pairs
requested_pairs = [
    # Top 3 (For Combined Plot & Singles)
    ('t(min)', 'T(°C)'),
    ('t(min)', 'SO2'),
    ('T(°C)', 'SO2'),
    
    # Others (Singles only)
    ('t(min)', 'CHg'),
    ('T(°C)', 'O2'),
    ('SO2', 'V1'),
    ('SO2', 'E1'),
    ('SO2', 'E2'),
    ('GHSV', 'T(°C)'),
    ('H2O', 'O2'),
    ('HCl', 'T(°C)'),
    ('HCl', 'SO2'),
    ('HCl', 'O2'),
    ('NO', 'O2'),
    ('NO', 'HCl'),
    ('CHg', 't(min)')
]

# Helper to match input names with actual DataFrame columns
available_features = X_train_display.columns.tolist()

def match_col(name):
    # Try exact match
    if name in available_features: return name
    # Try replacing symbols
    name_fixed = name.replace('℃', '°C')
    if name_fixed in available_features: return name_fixed
    name_raw = name.replace('°C', '℃')
    if name_raw in available_features: return name_raw
    # Fuzzy search
    for col in available_features:
        if name in col: return col
    return None

# ==========================================
# 3. Part A: Top 3 Combined Plot (1x3)
# ==========================================
print("\n2. Generating Top 3 Combined Plot...")

fig = plt.figure(figsize=(18, 5.5))
top3_list = requested_pairs[:3]

for i, (req_f1, req_f2) in enumerate(top3_list):
    f1 = match_col(req_f1)
    f2 = match_col(req_f2)
    
    if not f1 or not f2: continue
        
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    
    # Calculate
    pdp = partial_dependence(model, X_train_display, [f1, f2], kind='average', grid_resolution=30)
    
    if 'grid_values' in pdp:
        x1, x2 = pdp['grid_values'][0], pdp['grid_values'][1]
    else:
        x1, x2 = pdp['values'][0], pdp['values'][1]
    z = pdp['average'][0].T
    
    X1, X2 = np.meshgrid(x1, x2)
    
    # Plot Surface (Coolwarm + Mesh lines)
    surf = ax.plot_surface(X1, X2, z, cmap=cm.coolwarm, 
                           linewidth=0.3, edgecolors='k', alpha=0.9, antialiased=True)
    
    ax.set_xlabel(f1, fontsize=LABEL_SZ, fontweight='bold', labelpad=10)
    ax.set_ylabel(f2, fontsize=LABEL_SZ, fontweight='bold', labelpad=10)
    ax.set_zlabel('RHg (%)', fontsize=LABEL_SZ, fontweight='bold', rotation=90, labelpad=5)
    ax.tick_params(axis='both', which='major', labelsize=TICK_SZ)
    ax.view_init(elev=25, azim=-45)
    
    # Colorbar
    cbar = fig.colorbar(surf, ax=ax, shrink=0.5, aspect=12, pad=0.1)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'PDP_3D_Interactions_Top3_Combined.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 4. Part B: All Individual Plots (Loop)
# ==========================================
print("\n3. Generating Individual Plots for ALL requested pairs...")

for req_f1, req_f2 in requested_pairs:
    f1 = match_col(req_f1)
    f2 = match_col(req_f2)
    
    if not f1 or not f2:
        print(f"   [Skipped] Could not find columns for: {req_f1} vs {req_f2}")
        continue
        
    print(f"   Plotting: {f1} vs {f2}...")
    
    # Create Single Figure (High Res)
    fig = plt.figure(figsize=(8, 7))
    ax = fig.add_subplot(111, projection='3d')
    
    # Higher resolution for single plots
    pdp = partial_dependence(model, X_train_display, [f1, f2], kind='average', grid_resolution=40)
    
    if 'grid_values' in pdp:
        x1, x2 = pdp['grid_values'][0], pdp['grid_values'][1]
    else:
        x1, x2 = pdp['values'][0], pdp['values'][1]
    z = pdp['average'][0].T
    
    X1, X2 = np.meshgrid(x1, x2)
    
    # Plot
    surf = ax.plot_surface(X1, X2, z, cmap=cm.coolwarm, 
                           linewidth=0.3, edgecolors='k', alpha=0.95, antialiased=True)
    
    # Styling
    ax.set_xlabel(f1, fontsize=14, fontweight='bold', labelpad=12)
    ax.set_ylabel(f2, fontsize=14, fontweight='bold', labelpad=12)
    ax.set_zlabel('Predicted RHg (%)', fontsize=14, fontweight='bold', rotation=90, labelpad=8)
    ax.tick_params(axis='both', which='major', labelsize=11)
    
    # Optimal View Angle
    ax.view_init(elev=30, azim=-50) 
    
    # Colorbar
    cbar = fig.colorbar(surf, ax=ax, shrink=0.6, aspect=15, pad=0.1)
    cbar.ax.tick_params(labelsize=11)
    
    # Save Filename (Sanitized)
    safe_f1 = f1.replace('(', '').replace(')', '').replace('°C', 'C').replace('/', '')
    safe_f2 = f2.replace('(', '').replace(')', '').replace('°C', 'C').replace('/', '')
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'PDP_3D_{safe_f1}_vs_{safe_f2}.png', dpi=DPI, bbox_inches='tight')
    plt.close()

print(f"\nProcessing Complete. {len(requested_pairs)} pairs processed.")


In [ ]:
# ==========================================
# CELL 12: Accumulated Local Effects (ALE) Analysis
# ==========================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import warnings

warnings.filterwarnings("default")

# --- Visualization settings ---
DPI = 300
FONT_FAMILY = 'Arial'
LABEL_SZ = 14
TICK_SZ = 12
LINE_COLOR = '#000080' # Navy Blue (Same as PDP)
RUG_COLOR = '#2b7bba'  # Steel Blue

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.5 
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

# ==========================================
# 1. Load Config & Retrain
# ==========================================
print("1. Loading Config & Retraining Model...")
best_config_path = RESULTS_DIR / "best_config_info.txt"
best_params_path = RESULTS_DIR / "best_hyperparams.json"
if not best_config_path.exists() or not best_params_path.exists():
    raise RuntimeError("Run the sensitivity and hyperparameter optimization cells first.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_split_ratio = float(content[1])
with best_params_path.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=best_split_ratio, random_state=RANDOM_SEED
)

# Fix Feature Names
X_train_display = X_train.copy()
new_cols = {col: col.replace('℃', '°C') for col in X_train_display.columns}
X_train_display.rename(columns=new_cols, inplace=True)

model = RandomForestRegressor(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train_display, y_train)

# ==========================================
# 2. Get Top 9 Features
# ==========================================
feature_importance_path = RESULTS_DIR / "final_feature_importance.csv"
if not feature_importance_path.exists():
    raise RuntimeError("Run the final evaluation and feature-importance cell first.")
fi_df = pd.read_csv(feature_importance_path)
fi_df["Feature"] = fi_df["Feature"].str.replace("℃", "°C", regex=False)
top_9_features = fi_df["Feature"].head(9).tolist()
print(f"Top 9 Features: {top_9_features}")

# ==========================================
# 3. Custom ALE Calculation Function
# ==========================================
def compute_1d_ale(model, X, feature, n_bins=50):
    """
    Computes 1D Accumulated Local Effects (ALE) for regression.
    """
    X_data = X[feature].values
    # Find unique quantiles to avoid empty bins
    quantiles = np.unique(np.quantile(X_data, np.linspace(0, 1, n_bins+1)))
    n_bins_actual = len(quantiles) - 1
    
    # Define intervals (bins)
    # Centers of bins for plotting
    centers = (quantiles[:-1] + quantiles[1:]) / 2
    
    ale = np.zeros(n_bins_actual)
    
    # Calculate local effects
    for i in range(n_bins_actual):
        lower, upper = quantiles[i], quantiles[i+1]
        
        # Find samples in this bin
        in_bin = (X_data >= lower) & (X_data <= upper)
        
        if not np.any(in_bin):
            continue
            
        # Create modified datasets
        X_subset = X[in_bin].copy()
        
        # Lower bound prediction
        X_subset[feature] = lower
        pred_lower = model.predict(X_subset)
        
        # Upper bound prediction
        X_subset[feature] = upper
        pred_upper = model.predict(X_subset)
        
        # Average difference
        ale[i] = np.mean(pred_upper - pred_lower)
        
    # Accumulate
    ale = np.cumsum(ale)
    
    # Center the plot (ALE should have mean 0)
    # Weighted by number of samples in each bin is more accurate, 
    # but simple centering is standard for visualization.
    ale -= ale.mean()
    
    # Insert start point (optional, helps visualization)
    grid = np.concatenate([[quantiles[0]], quantiles[1:]])
    ale_curve = np.concatenate([[0], ale]) # Start from 0 relative
    ale_curve -= ale_curve.mean() # Re-center
    
    return grid, ale_curve

# ==========================================
# 4. Plot A: Combined 3x3 Grid
# ==========================================
print("\n2. Generating Combined ALE Plot (3x3)...")

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feature in enumerate(top_9_features):
    ax = axes[i]
    
    # Calculate ALE
    grid, ale = compute_1d_ale(model, X_train_display, feature, n_bins=40)
    
    # Plot Line
    ax.plot(grid, ale, color=LINE_COLOR, linewidth=2.5)
    
    # Add Rug Plot (Data Density)
    # Subsample if too many points to avoid solid block of color
    rug_data = X_train_display[feature].values
    if len(rug_data) > 500:
        rug_data = rng.choice(rug_data, 500, replace=False)
        
    ax.plot(rug_data, [np.min(ale)] * len(rug_data), '|', color=RUG_COLOR, alpha=0.3, markersize=8)
    
    # Styling
    ax.set_xlabel(feature, fontsize=LABEL_SZ, fontweight='bold')
    ax.set_ylabel("ALE Value", fontsize=LABEL_SZ, fontweight='bold')
    ax.tick_params(axis='both', which='major', labelsize=TICK_SZ)
    
    # Boxed Spines (Thick)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')
        
    ax.grid(False) # No grid

plt.tight_layout()
plt.subplots_adjust(hspace=0.4, wspace=0.3)
plt.savefig(RESULTS_DIR / 'ALE_Combined_Top9.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 5. Plot B: Individual Plots
# ==========================================
print("\n3. Generating Individual ALE Plots...")

for feature in top_9_features:
    fig, ax = plt.subplots(figsize=(6, 5))
    
    # Calculate
    grid, ale = compute_1d_ale(model, X_train_display, feature, n_bins=50) # Higher res for single
    
    # Plot
    ax.plot(grid, ale, color=LINE_COLOR, linewidth=3)
    
    # Rug
    rug_data = X_train_display[feature].values
    ax.plot(rug_data, [np.min(ale)] * len(rug_data), '|', color=RUG_COLOR, alpha=0.3, markersize=10)
    
    # Style
    ax.set_xlabel(feature, fontsize=16, fontweight='bold')
    ax.set_ylabel("ALE Value", fontsize=16, fontweight='bold')
    ax.tick_params(labelsize=14)
    ax.grid(False)
    
    # Box
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    
    # Save
    safe_name = feature.replace('(', '').replace(')', '').replace('°C', 'C').replace('/', '_')
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'ALE_Single_{safe_name}.png', dpi=DPI, bbox_inches='tight')
    plt.close()

print(f"All ALE plots generated successfully.")


In [ ]:
# ==========================================
# CELL 13: Scenario-Based Material Inverse Design (Strict Non-Zero & Robustness)
# ==========================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import warnings

warnings.filterwarnings("default")

# --- Visualization settings ---
DPI = 300
FONT_FAMILY = 'Arial'
LABEL_SZ = 16
TICK_SZ = 14

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.5 
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

# ==========================================
# 1. Setup & Train Model
# ==========================================
print("1. Initialization & Training...")
best_config_path = RESULTS_DIR / "best_config_info.txt"
best_params_path = RESULTS_DIR / "best_hyperparams.json"
if not best_config_path.exists() or not best_params_path.exists():
    raise RuntimeError("Run the sensitivity and hyperparameter optimization cells first.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_split_ratio = float(content[1])
with best_params_path.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=best_split_ratio, random_state=RANDOM_SEED)

# Fix Names
X_train_display = X_train.copy()
new_cols = {col: col.replace('℃', '°C') for col in X_train_display.columns}
X_train_display.rename(columns=new_cols, inplace=True)

model = RandomForestRegressor(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train_display, y_train)

# ==========================================
# 2. Define Feature Groups & Constraints
# ==========================================
# Material Components (First 11 columns based on description)
# Es, CA, Ds, Ms, E1, C1, D1, M1, V1, E2, C2
material_cols = ['Es', 'CA', 'Ds', 'Ms', 'E1', 'C1', 'D1', 'M1', 'V1', 'E2', 'C2']

# Identify other columns
all_cols = X_train_display.columns.tolist()
t_col = next(c for c in all_cols if 'T(' in c)
so2_col = next(c for c in all_cols if 'SO2' in c)

# Calculate Non-Zero Ranges from Training Data
# This ensures we don't predict impossible values, but forces > 0
bounds = {}
for col in material_cols:
    if col in X_train_display.columns:
        # Filter out 0 and negative values to find true physical minimum
        non_zero_vals = X_train_display[col][X_train_display[col] > 0.0001]
        if len(non_zero_vals) > 0:
            min_v = non_zero_vals.min()
            max_v = non_zero_vals.max()
        else:
            # Fallback if a column is all 0s (unlikely based on your request)
            min_v = 0.01 
            max_v = 1.0
        bounds[col] = (min_v, max_v)

print("   [Constraint] Enforcing non-zero values for all material components.")
print(f"   [Constraint] Mass Balance: CA = C1 + C2")

# ==========================================
# 3. Simulation Function (Robustness Check)
# ==========================================
def find_optimal_recipe(temp_target, n_trials=100000):
    print(f"\n>> Optimizing for T={temp_target}°C (High SO2 + Zero SO2 Check)...")
    
    # --- A. Generate Random Material Recipes (Cols 1-11) ---
    virtual_df = pd.DataFrame()
    
    # 1. Independent Sampling for Metals (Es, Ds, Ms, E1, D1, M1, V1, E2)
    # We allow them to vary independently within valid ranges to find "Theoretical Best"
    # Logic: Uniform sampling between min (non-zero) and max
    independent_cols = ['Es', 'Ds', 'Ms', 'E1', 'D1', 'M1', 'V1', 'E2']
    for col in independent_cols:
        low, high = bounds[col]
        virtual_df[col] = rng.uniform(low, high, n_trials)
        
    # 2. Mass Balance for C (C1, C2 -> CA)
    # Sample C1 and C2, calculate CA
    c1_low, c1_high = bounds['C1']
    c2_low, c2_high = bounds['C2']
    
    virtual_df['C1'] = rng.uniform(c1_low, c1_high, n_trials)
    virtual_df['C2'] = rng.uniform(c2_low, c2_high, n_trials)
    virtual_df['CA'] = virtual_df['C1'] + virtual_df['C2']
    
    # --- B. Handle Ignored Features (Cols 12-17 + Others) ---
    # Fix to Mean/Median of Training Data (Representing "Average Structure/Conditions")
    defaults = X_train_display.mean().to_dict()
    
    # Fill missing columns with defaults
    for col in all_cols:
        if col not in virtual_df.columns:
            virtual_df[col] = defaults[col]
    
    # --- C. Apply Process Constraints (Target Scenario) ---
    # 1. Temperature
    virtual_df[t_col] = temp_target
    
    # 2. SO2 (High Concentration Scenario)
    # Randomly sample between 0.0005 and 0.001
    virtual_df[so2_col] = rng.uniform(0.0005, 0.001, n_trials)
    
    # 3. m (Mass constraint 0.1 - 0.5)
    virtual_df['m'] = rng.uniform(0.1, 0.5, n_trials)
    
    # Ensure Column Order
    virtual_df = virtual_df[all_cols]
    
    # --- D. Prediction & Filtering ---
    # Predict Efficiency under High SO2
    pred_high_so2 = model.predict(virtual_df)
    
    # Create a check dataframe for Zero SO2
    check_df = virtual_df.copy()
    check_df[so2_col] = 0.0 # Set SO2 to 0
    pred_zero_so2 = model.predict(check_df)
    
    # --- E. Selection Strategy ---
    # We want: Maximize (High SO2 Perf) SUBJECT TO (Zero SO2 Perf > Threshold)
    
    # Define "Good enough" for Zero SO2 (e.g., > 85% or Top 20% of predictions)
    # Let's be strict: Must be > 80% removal even without SO2
    robust_indices = np.where(pred_zero_so2 > 80)[0]
    
    if len(robust_indices) == 0:
        print("   Warning: No robust materials found. Relaxing Zero SO2 constraint.")
        robust_indices = np.arange(n_trials) # Fallback to all
        
    # Filter candidates
    candidates = virtual_df.iloc[robust_indices]
    scores = pred_high_so2[robust_indices]
    scores_zero = pred_zero_so2[robust_indices]
    
    # Find Best among robust candidates
    best_local_idx = np.argmax(scores)
    best_recipe = candidates.iloc[best_local_idx].copy()
    best_score_high = scores[best_local_idx]
    best_score_zero = scores_zero[best_local_idx]
    
    return best_recipe, best_score_high, best_score_zero

# ==========================================
# 4. Execute Optimization (100C & 150C)
# ==========================================
recipe_100, score_100_h, score_100_z = find_optimal_recipe(100)
recipe_150, score_150_h, score_150_z = find_optimal_recipe(150)

# Output Table
def print_nice_recipe(row, score_h, score_z, label):
    print(f"\n★ OPTIMAL RECIPE: {label}")
    print(f"   Pred RHg (High SO2): {score_h:.2f}%")
    print(f"   Pred RHg (Zero SO2): {score_z:.2f}% (Robustness Check Passed)")
    print("-" * 35)
    print(f"{'Component':<10} | {'Value':<10}")
    print("-" * 35)
    for c in material_cols:
        print(f"{c:<10} | {row[c]:.4f}")
    print("-" * 35)

print_nice_recipe(recipe_100, score_100_h, score_100_z, "T=100°C")
print_nice_recipe(recipe_150, score_150_h, score_150_z, "T=150°C")

# Save
res_df = pd.DataFrame([recipe_100, recipe_150])
res_df.index = ["Recipe_100C", "Recipe_150C"]
res_df.to_csv(RESULTS_DIR / "final_material_recipes.csv")

# ==========================================
# 5. Visualization: T vs SO2 (Best Recipe)
# ==========================================
print("\n4. Generating Performance Map (Best Recipe @ 150C)...")

# Use the best recipe from 150C case
target_recipe = recipe_150.copy()

plt.figure(figsize=(9, 7.5))
ax = plt.gca()

# Grid
t_range = np.linspace(50, 250, 100)
so2_range = np.linspace(0, 0.002, 100)
XX, YY = np.meshgrid(t_range, so2_range)

# Fill Grid
grid_df = pd.DataFrame([target_recipe] * 10000)
grid_df[t_col] = XX.ravel()
grid_df[so2_col] = YY.ravel()

# Predict
ZZ = model.predict(grid_df).reshape(XX.shape)

# Contour
levels = np.linspace(ZZ.min(), ZZ.max(), 25)
contour = ax.contourf(XX, YY, ZZ, levels=levels, cmap='RdYlBu_r', alpha=0.95)

# Highlight Optimal Zone (>90%)
cs = ax.contour(XX, YY, ZZ, levels=[90], colors='white', linewidths=2, linestyles='--')
# Label the contour line
ax.clabel(cs, inline=True, fontsize=10, fmt='>90%%')

# Add marker for the design point
design_so2_avg = 0.00075 # Avg of 0.0005-0.001
ax.scatter(150, design_so2_avg, c='black', s=100, marker='*', label='Design Point')

# Styling
ax.set_xlabel(t_col, fontsize=LABEL_SZ, fontweight='bold')
ax.set_ylabel(so2_col, fontsize=LABEL_SZ, fontweight='bold')
ax.tick_params(labelsize=TICK_SZ)
ax.set_title("Robustness Map: Best Designed Material", fontsize=LABEL_SZ, fontweight='bold', pad=15)

for spine in ax.spines.values():
    spine.set_linewidth(1.5)
    spine.set_color('black')

cbar = plt.colorbar(contour)
cbar.set_label('Predicted RHg (%)', fontsize=14, fontweight='bold')
cbar.ax.tick_params(labelsize=12)

plt.legend(loc='upper right', frameon=True, edgecolor='black')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'Design_Material_Robustness_Contour.png', dpi=DPI, bbox_inches='tight')
plt.show()

print("Inverse Design Completed.")


In [ ]:
# ==========================================
# CELL 14: Targeted Optimization (C1/C2 Search under Constraints)
# ==========================================

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import warnings

warnings.filterwarnings("default")

# --- Visualization Style ---
DPI = 300
FONT_FAMILY = 'Arial'
LABEL_SZ = 14
TICK_SZ = 12

plt.rcParams['font.family'] = FONT_FAMILY
plt.rcParams['axes.linewidth'] = 1.5 
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'

# ==========================================
# 1. Setup & Train Model
# ==========================================
print("1. Initialization...")
best_config_path = RESULTS_DIR / "best_config_info.txt"
best_params_path = RESULTS_DIR / "best_hyperparams.json"
if not best_config_path.exists() or not best_params_path.exists():
    raise RuntimeError("Run the sensitivity and hyperparameter optimization cells first.")
with best_config_path.open("r", encoding="utf-8") as f:
    content = f.read().strip().split(",")
if len(content) != 2:
    raise ValueError(f"Invalid configuration file: {best_config_path}")
best_split_ratio = float(content[1])
with best_params_path.open("r", encoding="utf-8") as f:
    best_params = json.load(f)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=best_split_ratio, random_state=RANDOM_SEED)

# Fix Names
X_train_display = X_train.copy()
new_cols = {col: col.replace('℃', '°C') for col in X_train_display.columns}
X_train_display.rename(columns=new_cols, inplace=True)

model = RandomForestRegressor(**best_params, random_state=RANDOM_SEED, n_jobs=-1)
model.fit(X_train_display, y_train)

# ==========================================
# 2. Define Constraints & Lookup Physics
# ==========================================
# Target Constants
TARGET_ES = 1.54
TARGET_E1 = 1.55
TARGET_E2_LIST = [1.88, 1.91]
TARGET_T_LIST = [100, 150]
TARGET_SO2_RANGE = (0.0005, 0.001)

# Helper: Find physical properties (Ds, Ms, D1, M1) associated with Es and E1
# This ensures we don't invent a fake metal.
def get_associated_properties(df, target_val, col_name, assoc_cols):
    # Find rows close to the target value (handling float precision)
    matches = df[np.isclose(df[col_name], target_val, atol=0.01)]
    if len(matches) == 0:
        print(f"Warning: No exact match for {col_name}={target_val}. Using global mean.")
        return [df[c].mean() for c in assoc_cols]
    # Return the mode (most common) or mean of the associated properties
    return [matches[c].mean() for c in assoc_cols]

# Lookup Properties
# Es=1.54 -> Get Ds, Ms
ds_val, ms_val = get_associated_properties(X_train_display, TARGET_ES, 'Es', ['Ds', 'Ms'])
# E1=1.55 -> Get D1, M1
d1_val, m1_val = get_associated_properties(X_train_display, TARGET_E1, 'E1', ['D1', 'M1'])

print(f"\n[Constraint Check]")
print(f"   Metal A (Es={TARGET_ES}): Ds={ds_val:.2f}, Ms={ms_val:.2f}")
print(f"   Metal B (E1={TARGET_E1}): D1={d1_val:.2f}, M1={m1_val:.2f}")

# ==========================================
# 3. Optimization Function (Grid Search C1/C2)
# ==========================================
def optimize_c1_c2(temp, e2_val):
    # 1. Create Grid for C1 and C2
    c1_range = np.linspace(X_train_display['C1'].min(), X_train_display['C1'].max(), 100)
    c2_range = np.linspace(X_train_display['C2'].min(), X_train_display['C2'].max(), 100)
    XX, YY = np.meshgrid(c1_range, c2_range)
    
    # Flatten for prediction
    flat_c1 = XX.ravel()
    flat_c2 = YY.ravel()
    n_points = len(flat_c1)
    
    # 2. Build Input DataFrame
    sim_df = pd.DataFrame()
    
    # A. Fill Defaults (Mean of everything)
    defaults = X_train_display.mean().to_dict()
    for col in X_train_display.columns:
        sim_df[col] = np.full(n_points, defaults[col])
        
    # B. Apply Fixed Constraints
    sim_df['Es'] = TARGET_ES
    sim_df['Ds'] = ds_val
    sim_df['Ms'] = ms_val
    
    sim_df['E1'] = TARGET_E1
    sim_df['D1'] = d1_val
    sim_df['M1'] = m1_val
    
    sim_df['E2'] = e2_val
    
    # Temperature
    t_col = next(c for c in X_train_display.columns if 'T(' in c)
    sim_df[t_col] = temp
    
    # SO2 (Average of high range)
    so2_col = next(c for c in X_train_display.columns if 'SO2' in c)
    so2_avg = (TARGET_SO2_RANGE[0] + TARGET_SO2_RANGE[1]) / 2
    sim_df[so2_col] = so2_avg
    
    # C. Apply Variables
    sim_df['C1'] = flat_c1
    sim_df['C2'] = flat_c2
    sim_df['CA'] = flat_c1 + flat_c2 # Dependent variable
    
    # 3. Predict
    # Ensure column order
    sim_df = sim_df[X_train_display.columns]
    preds = model.predict(sim_df)
    
    # 4. Find Best
    best_idx = np.argmax(preds)
    best_c1 = flat_c1[best_idx]
    best_c2 = flat_c2[best_idx]
    best_score = preds[best_idx]
    
    return XX, YY, preds.reshape(XX.shape), (best_c1, best_c2, best_score)

# ==========================================
# 4. Run & Visualize
# ==========================================
scenarios = [
    (100, 1.88), (100, 1.91),
    (150, 1.88), (150, 1.91)
]

print("\nRunning Optimization for 4 Scenarios...")
results = []

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for i, (t, e2) in enumerate(scenarios):
    ax = axes[i]
    
    # Run Optimization
    XX, YY, Z, best = optimize_c1_c2(t, e2)
    best_c1, best_c2, best_score = best
    
    results.append({
        'Temp': t,
        'E2': e2,
        'Best_C1': best_c1,
        'Best_C2': best_c2,
        'Best_CA': best_c1 + best_c2,
        'Pred_RHg': best_score
    })
    
    # Plot Contour
    # EST Style: Blue -> Yellow -> Red
    levels = np.linspace(Z.min(), Z.max(), 20)
    contour = ax.contourf(XX, YY, Z, levels=levels, cmap='RdYlBu_r', alpha=0.9)
    
    # Highlight Best Point
    ax.scatter(best_c1, best_c2, c='white', s=100, marker='*', edgecolors='black', label=f'Max: {best_score:.1f}%')
    
    # Add CA diagonal lines (Optional but helpful for interpretation)
    # CA = C1 + C2 -> C2 = -C1 + CA
    # We can plot a few reference lines if needed, but color already shows pattern
    
    # Styling
    ax.set_title(f"Scenario: T={t}°C, E2={e2}", fontsize=LABEL_SZ, fontweight='bold')
    ax.set_xlabel('C1 Concentration', fontsize=12, fontweight='bold')
    ax.set_ylabel('C2 Concentration', fontsize=12, fontweight='bold')
    ax.legend(loc='upper right', frameon=True, edgecolor='black')
    
    # Boxed
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
        spine.set_color('black')

# Colorbar (Shared or Individual? Individual is safer for different ranges)
# Let's add one common colorbar or individual? Individual is better here.
# We will just let the colors speak for relative difference within plot.

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'Targeted_Optimization_C1_C2.png', dpi=DPI, bbox_inches='tight')
plt.show()

# ==========================================
# 5. Output Text Report
# ==========================================
res_df = pd.DataFrame(results)
res_df = res_df.sort_values(by='Pred_RHg', ascending=False)

print("\n" + "="*60)
print("★ TARGETED OPTIMIZATION RESULTS (High SO2 Condition)")
print("="*60)
print(f"{'Temp':<6} | {'E2':<6} | {'Best C1':<10} | {'Best C2':<10} | {'Best CA':<10} | {'Pred RHg':<10}")
print("-" * 60)
for _, row in res_df.iterrows():
    print(f"{row['Temp']:<6.0f} | {row['E2']:<6.2f} | {row['Best_C1']:<10.4f} | {row['Best_C2']:<10.4f} | {row['Best_CA']:<10.4f} | {row['Pred_RHg']:<10.2f}%")
print("-" * 60)

res_df.to_csv(RESULTS_DIR / 'targeted_optimization_results.csv', index=False)
print("Results saved to 'targeted_optimization_results.csv'")
